In [ ]:
import pandas as pd

# 📄 Cargar el archivo principal
ruta_txt = r"C:\MICHAEL\AUTOMATIZACION DE LABORES\INSERCION DE  UBIGE\inicial-EDNA25T4.txt"
ruta_xlsx = r"C:\MICHAEL\AUTOMATIZACION DE LABORES\INSERCION DE  UBIGE\ubi.xlsx"
anchos = [3, 4, 2, 10, 6, 2, 4, 4, 4, 8, 8, 8, 18, 14, 14]

# 📄 Leer el archivo y convertir todos los campos a texto desde el inicio
df = pd.read_fwf(
    ruta_txt,
    widths=anchos,
    header=None,
    dtype=str  # 👈 Forzar que todas las columnas se lean como texto
)

# ✅ Asignar nombres de columnas
df.columns = [
    "CodEmpresa", "Año", "Semestre", "Suministro", "Ubigeo", "Tensión",
    "Int_NO_PROG", "Int_MANT", "Int_EXP",
    "Dur_NO_PROG", "Dur_MANT", "Dur_EXP",
    "ERS_kWh", "Comp_Punto_USD", "Comp_Ley_USD"
]

display(df)
#Etapa 1
# 📄 Cargar el archivo auxiliar con relación Suministro–Ubigeo

df_ubi = pd.read_excel(ruta_xlsx, dtype=str)
df_ubi = df_ubi.drop_duplicates(subset=["Suministro", "Ubigeo"])

# 🔍 Rellenar los ubigeos faltantes

def completar_ubigeo(row):
    # Completar Ubigeo vacío
    if pd.isna(row["Ubigeo"]) or str(row["Ubigeo"]).strip() == "":
        resultado = df_ubi[df_ubi["Suministro"] == row["Suministro"]]
        ubigeo = resultado["Ubigeo"].values[0] if not resultado.empty else row["Ubigeo"]
    else:
        ubigeo = row["Ubigeo"]
   
    # Formatear a exactamente 6 dígitos
    if pd.notna(ubigeo):
        # Convertir a string y limpiar
        ubigeo_str = str(ubigeo).strip()
       
        # Si tiene punto decimal, tomar solo la parte entera
        if '.' in ubigeo_str:
            ubigeo_str = ubigeo_str.split('.')[0]
       
        # Solo dígitos
        if ubigeo_str.isdigit():
            # Tomar los últimos 6 dígitos (si es más largo) o rellenar con ceros
            ubigeo_str = ubigeo_str[-6:]  # Tomar últimos 6 dígitos
            return ubigeo_str.zfill(6)    # Rellenar con ceros a la izquierda si es necesario
        else:
            return ubigeo_str
   
    return ubigeo

# 📄 Mostrar resultado
display(df)
# ✅ Convertir todos los valores a string
# 🧾 Aplicar formato a todas las filas

# 🧠 Detectar filas que cumplen la condición antes de modificar
cond_mant = (df["Int_MANT"] == "1") & (df["Dur_MANT"] == "000")
cond_exp = (df["Int_EXP"] == "1") & (df["Dur_EXP"] == "000")
cond_mant_2 = (df["Int_MANT"] == "0") & (df["Dur_MANT"] != "000")
cond_exp_2 = (df["Int_EXP"] == "0") & (df["Dur_EXP"] != "000")

# 🧾 Filas afectadas
filas_mant_antes = df[cond_mant].copy()
filas_exp_antes = df[cond_exp].copy()
filas_mant_2_antes = df[cond_mant_2].copy()
filas_exp_2_antes = df[cond_exp_2].copy()

print("\n🔍 FILAS A CORREGIR:")
print(f"Filas con Int_MANT=1 y Dur_MANT=000: {len(filas_mant_antes)}")
print(f"Filas con Int_EXP=1 y Dur_EXP=000: {len(filas_exp_antes)}")
print(f"Filas con Int_MANT=0 y Dur_MANT!=000: {len(filas_mant_2_antes)}")
print(f"Filas con Int_EXP=0 y Dur_EXP!=000: {len(filas_exp_2_antes)}")

if len(filas_mant_antes) > 0:
    print("\n🔍 Ejemplos de filas con Int_MANT=1 y Dur_MANT=000:")
    print(filas_mant_antes[["Suministro", "Int_MANT", "Dur_MANT"]].head())

# 🔧 Aplicar correcciones
df.loc[cond_mant, "Int_MANT"] = "0"
df.loc[cond_exp, "Int_EXP"] = "0"
df.loc[cond_mant_2, "Dur_MANT"] = "000"
df.loc[cond_exp_2, "Dur_EXP"] = "000"

print("\n✅ Correcciones aplicadas")



# 1️⃣ Cargar el Excel con los valores corregidos
df_tension = pd.read_excel(
    r"C:\MICHAEL\AUTOMATIZACION DE LABORES\INSERCION DE  UBIGE\tarifa.xlsx",
    sheet_name="Hoja1",
    dtype=str
)
df_tension = df_tension.drop_duplicates(subset=["Suministro", "tension"])

# 2️⃣ Guardar columna original antes del reemplazo
df["Tensión_original"] = df["Tensión"]

# 3️⃣ Merge para traer tensiones nuevas
df_actualizado = df.merge(
    df_tension[["Suministro", "tension"]],
    on="Suministro",
    how="left"
)

# 4️⃣ Reemplazar solo si hay nuevo valor
df_actualizado["Tensión"] = df_actualizado["tension"].fillna(df_actualizado["Tensión_original"])

# 5️⃣ Eliminar columna auxiliar
df_actualizado = df_actualizado.drop(columns=["tension"])

# 6️⃣ Comparar valores y marcar cambios
df_actualizado["Cambio_Tensión"] = df_actualizado["Tensión"] != df_actualizado["Tensión_original"]

# 7️⃣ Mostrar filas que cambiaron
filas_modificadas = df_actualizado[df_actualizado["Cambio_Tensión"]]


display(df_actualizado)
print("🔍 Filas con cambios en la columna 'Tensión':")
display(filas_modificadas)
###########################################################################################

# 📤 Guardar en archivo .txt
with open(ruta_salida, "w", encoding="utf-8") as f:
    for _, row in df.iterrows():
        linea = "".join(str(row[col]).rjust(width)[:width] for col, width in zip(df.columns, anchos))
        f.write(linea + "\n")



print("✅ Archivo exportado correctamente en formato de ancho fijo.")

ruta_salida = r"C:\MICHAEL\AUTOMATIZACION DE LABORES\INSERCION DE  UBIGE\EDNA25S1_Modificado.txt"

# 📤 Guardar en archivo .txt
with open(ruta_salida, "w", encoding="utf-8") as f:
    for _, row in df_actualizado.iterrows():
        linea = "".join(str(row[col]).rjust(width)[:width] for col, width in zip(df_actualizado.columns, anchos))
        f.write(linea + "\n")



print("✅ Archivo exportado correctamente en formato de ancho fijo.")


,CodEmpresa,Año,Semestre,Suministro,Ubigeo,Tensión,Int_NO_PROG,Int_MANT,Int_EXP,Dur_NO_PROG,Dur_MANT,Dur_EXP,ERS_kWh,Comp_Punto_USD,Comp_Ley_USD
0,EDN,2024,S2,521895,150135,BT,3,1,0,1181,000,000,1387000,15374,17066
1,EDN,2024,S2,521896,150135,BT,3,0,0,1459,000,000,693000,11736,12581
2,EDN,2024,S2,521897,150135,BT,4,0,0,2504,000,000,606000,30295,19099
3,EDN,2024,S2,521902,NaN,BT,3,0,0,1181,000,000,738000,08180,08133
4,EDN,2024,S2,521908,150135,BT,3,0,0,3074,000,000,687000,51825,25669
5,EDN,2024,S2,521909,150135,BT,3,1,0,3074,000,000,534000,40283,19963
6,EDN,2024,S2,521911,150135,BT,3,0,1,3074,000,000,3218000,242755,141288
7,EDN,2024,S2,521912,150135,BT,3,0,0,3074,000,000,1992000,150269,78347
8,EDN,2024,S2,521913,150135,BT,3,0,0,3074,000,000,795000,59972,25632
9,EDN,2024,S2,521914,150135,BT,3,0,0,3074,000,000,3437000,259276,126447


,CodEmpresa,Año,Semestre,Suministro,Ubigeo,Tensión,Int_NO_PROG,Int_MANT,Int_EXP,Dur_NO_PROG,Dur_MANT,Dur_EXP,ERS_kWh,Comp_Punto_USD,Comp_Ley_USD
0,EDN,2024,S2,521895,150135,BT,3,1,0,1181,000,000,1387000,15374,17066
1,EDN,2024,S2,521896,150135,BT,3,0,0,1459,000,000,693000,11736,12581
2,EDN,2024,S2,521897,150135,BT,4,0,0,2504,000,000,606000,30295,19099
3,EDN,2024,S2,521902,150135,BT,3,0,0,1181,000,000,738000,08180,08133
4,EDN,2024,S2,521908,150135,BT,3,0,0,3074,000,000,687000,51825,25669
5,EDN,2024,S2,521909,150135,BT,3,1,0,3074,000,000,534000,40283,19963
6,EDN,2024,S2,521911,150135,BT,3,0,1,3074,000,000,3218000,242755,141288
7,EDN,2024,S2,521912,150135,BT,3,0,0,3074,000,000,1992000,150269,78347
8,EDN,2024,S2,521913,150135,BT,3,0,0,3074,000,000,795000,59972,25632
9,EDN,2024,S2,521914,150135,BT,3,0,0,3074,000,000,3437000,259276,126447


🔍 Filas con Int_MANT = 1 y Dur_MANT = 0 (antes de corrección):


,Suministro,Int_MANT,Dur_MANT
0,521895,1,000
5,521909,1,000
14,521921,1,000


🔍 Filas con Int_EXP = 1 y Dur_EXP = 0 (antes de corrección):


,Suministro,Int_EXP,Dur_EXP
6,521911,1,000
10,521915,1,000
16,521925,1,000


... (4 outputs left out)

,CodEmpresa,Año,Semestre,Suministro,Ubigeo,Tensión,Int_NO_PROG,Int_MANT,Int_EXP,Dur_NO_PROG,Dur_MANT,Dur_EXP,ERS_kWh,Comp_Punto_USD,Comp_Ley_USD,Tensión_original,Cambio_Tensión
0,EDN,2024,S2,521895,150135,AT,3,0,0,1181,000,000,1387000,15374,17066,BT,True
1,EDN,2024,S2,521896,150135,AT,3,0,0,1459,000,000,693000,11736,12581,BT,True
2,EDN,2024,S2,521897,150135,AT,4,0,0,2504,000,000,606000,30295,19099,BT,True
3,EDN,2024,S2,521902,150135,AT,3,0,0,1181,000,000,738000,08180,08133,BT,True
4,EDN,2024,S2,521908,150135,BT,3,0,0,3074,000,000,687000,51825,25669,BT,False
5,EDN,2024,S2,521909,150135,BT,3,0,0,3074,000,000,534000,40283,19963,BT,False
6,EDN,2024,S2,521911,150135,BT,3,0,0,3074,000,000,3218000,242755,141288,BT,False
7,EDN,2024,S2,521912,150135,BT,3,0,0,3074,000,000,1992000,150269,78347,BT,False
8,EDN,2024,S2,521913,150135,BT,3,0,0,3074,000,000,795000,59972,25632,BT,False
9,EDN,2024,S2,521914,150135,BT,3,0,0,3074,000,000,3437000,259276,126447,BT,False


🔍 Filas con cambios en la columna 'Tensión':


,CodEmpresa,Año,Semestre,Suministro,Ubigeo,Tensión,Int_NO_PROG,Int_MANT,Int_EXP,Dur_NO_PROG,Dur_MANT,Dur_EXP,ERS_kWh,Comp_Punto_USD,Comp_Ley_USD,Tensión_original,Cambio_Tensión
0,EDN,2024,S2,521895,150135,AT,3,0,0,1181,000,000,1387000,15374,17066,BT,True
1,EDN,2024,S2,521896,150135,AT,3,0,0,1459,000,000,693000,11736,12581,BT,True
2,EDN,2024,S2,521897,150135,AT,4,0,0,2504,000,000,606000,30295,19099,BT,True
3,EDN,2024,S2,521902,150135,AT,3,0,0,1181,000,000,738000,08180,08133,BT,True


✅ Archivo exportado correctamente en formato de ancho fijo.
✅ Archivo exportado correctamente en formato de ancho fijo.


,ID,CodEmpresa,Año,Semestre,Suministro,Ubigeo,Tensión,Int_NO_PROG,Int_MANT,Int_EXP,Dur_NO_PROG,Dur_MANT,Dur_EXP,ERS_kWh,Comp_Punto_USD,Comp_Ley_USD
0,1,EDN,2024,S2,521895,150135,AT,3,0,0,1181,000,000,1387000,15374,17066
1,2,EDN,2024,S2,521896,150135,AT,3,0,0,1459,000,000,693000,11736,12581
2,3,EDN,2024,S2,521897,150135,AT,4,0,0,2504,000,000,606000,30295,19099
3,4,EDN,2024,S2,521902,150135,AT,3,0,0,1181,000,000,738000,08180,08133
4,5,EDN,2024,S2,521908,150135,BT,3,0,0,3074,000,000,687000,51825,25669
5,6,EDN,2024,S2,521909,150135,BT,3,0,0,3074,000,000,534000,40283,19963
6,7,EDN,2024,S2,521911,150135,BT,3,0,0,3074,000,000,3218000,242755,141288
7,8,EDN,2024,S2,521912,150135,BT,3,0,0,3074,000,000,1992000,150269,78347
8,9,EDN,2024,S2,521913,150135,BT,3,0,0,3074,000,000,795000,59972,25632
9,10,EDN,2024,S2,521914,150135,BT,3,0,0,3074,000,000,3437000,259276,126447


,ID,CodEmpresa,Año,Semestre,Suministro,Ubigeo,Tensión,Int_NO_PROG,Int_MANT,Int_EXP,Dur_NO_PROG,Dur_MANT,Dur_EXP,ERS_kWh,Comp_Punto_USD,Comp_Ley_USD
1,2,EDN,2024,S2,521896,150135,AT,3,0,0,1459,000,000,693000,11736,12581
2,3,EDN,2024,S2,521897,150135,AT,4,0,0,2504,000,000,606000,30295,19099
3,4,EDN,2024,S2,521902,150135,AT,3,0,0,1181,000,000,738000,08180,08133
5,6,EDN,2024,S2,521909,150135,BT,3,0,0,3074,000,000,534000,40283,19963
6,7,EDN,2024,S2,521911,150135,BT,3,0,0,3074,000,000,3218000,242755,141288
7,8,EDN,2024,S2,521912,150135,BT,3,0,0,3074,000,000,1992000,150269,78347
8,9,EDN,2024,S2,521913,150135,BT,3,0,0,3074,000,000,795000,59972,25632
9,10,EDN,2024,S2,521914,150135,BT,3,0,0,3074,000,000,3437000,259276,126447
10,11,EDN,2024,S2,521915,150135,BT,3,0,0,3074,000,000,1413000,106592,51848
11,12,EDN,2024,S2,521917,150135,BT,3,0,0,3074,000,000,2346000,176974,88630


,ID,CodEmpresa,Año,Semestre,Suministro,Ubigeo,Tensión,Int_NO_PROG,Int_MANT,Int_EXP,Dur_NO_PROG,Dur_MANT,Dur_EXP,ERS_kWh,Comp_Punto_USD,Comp_Ley_USD
0,1,EDN,2024,S2,521895,150135,AT,3,0,0,1181,000,000,1387000,15374,17066
4,5,EDN,2024,S2,521908,150135,BT,3,0,0,3074,000,000,687000,51825,25669
14,15,EDN,2024,S2,521921,150135,BT,3,0,0,3074,000,000,1579000,119114,72207


,CodEmpresa,Año,Semestre,Suministro,Ubigeo,Tensión,Int_NO_PROG,Int_MANT,Int_EXP,Dur_NO_PROG,Dur_MANT,Dur_EXP,ERS_kWh,Comp_Punto_USD,Comp_Ley_USD
1,EDN,2024,S2,521896,150135,AT,3,0,0,1459,000,000,693000,11736,12581
2,EDN,2024,S2,521897,150135,AT,4,0,0,2504,000,000,606000,30295,19099
3,EDN,2024,S2,521902,150135,AT,3,0,0,1181,000,000,738000,08180,08133
5,EDN,2024,S2,521909,150135,BT,3,0,0,3074,000,000,534000,40283,19963
6,EDN,2024,S2,521911,150135,BT,3,0,0,3074,000,000,3218000,242755,141288
7,EDN,2024,S2,521912,150135,BT,3,0,0,3074,000,000,1992000,150269,78347
8,EDN,2024,S2,521913,150135,BT,3,0,0,3074,000,000,795000,59972,25632
9,EDN,2024,S2,521914,150135,BT,3,0,0,3074,000,000,3437000,259276,126447
10,EDN,2024,S2,521915,150135,BT,3,0,0,3074,000,000,1413000,106592,51848
11,EDN,2024,S2,521917,150135,BT,3,0,0,3074,000,000,2346000,176974,88630


✅ Archivo exportado correctamente en formato de ancho fijo.
